# recurrentlens 03 — Train a Mamba-130M SAE on Colab T4

Estimated wall-clock on Colab T4 (free tier): ~45 min for `n_tokens=5_000_000`, `n_steps=2000`, `d_sae=16384`.

This notebook is **how v0.1.1 pretrained artifacts get produced**. The maintainer can't run it (CPU-only build env); community runs are how the Hub registry gets seeded.

> **Before pushing to the Hub:** set `HF_TOKEN` in the Colab secrets panel and uncomment the `huggingface-cli login` cell. The token never leaves your runtime.

In [ ]:
%pip install -q 'recurrentlens[mamba]' || %pip install -q git+https://github.com/hinanohart/recurrentlens.git

In [ ]:
# Optional: log in to HF Hub if you want to push the trained SAE.
# from google.colab import userdata
# import os, subprocess
# os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
# subprocess.run(['huggingface-cli', 'login', '--token', os.environ['HF_TOKEN']], check=True)
import os
print('HF_TOKEN set:', bool(os.environ.get('HF_TOKEN')))

In [ ]:
import recurrentlens as rl

LAYER = 6  # try also 2, 10
VARIANT = 'topk'
D_SAE = 16384
N_TOKENS = 5_000_000
N_STEPS = 2000

model = rl.load_model('state-spaces/mamba-130m-hf')
print(model)

In [ ]:
sae = rl.train_sae(
    model,
    hook_site='out_proj_out',
    layer=LAYER,
    variant=VARIANT,
    d_sae=D_SAE,
    n_tokens=N_TOKENS,
    n_steps=N_STEPS,
    cache_backend='mmap',
    save_to=f'sae_mamba130m_L{LAYER}.safetensors',
    k=32,
)
print(sae)

In [ ]:
# Quick bench on a held-out batch.
from recurrentlens.sae.cache import ActivationCache
from recurrentlens.hooks.registry import HookManager
import torch

eval_cache = ActivationCache(capacity=32_768, d_in=model.d_model, backend='ram')
from recurrentlens.sae.datastream import make_token_iter_from_text, _stream_dataset_text
text_iter = _stream_dataset_text('HuggingFaceFW/fineweb-edu')
token_iter = make_token_iter_from_text(model.tokenizer, text_iter, seq_len=512, batch_size=4, device=model.device)
from recurrentlens.sae.datastream import tokens_to_activations
tokens_to_activations(model, hook_site='out_proj_out', layer=LAYER, token_iter=token_iter, cache=eval_cache, n_tokens=32_768)
acts = torch.as_tensor(eval_cache._store[: len(eval_cache)])
report = rl.bench.evaluate(sae, activations=acts)
print(report)

In [ ]:
# Push to Hub (requires HF_TOKEN).
# url = rl.hub.push_sae(sae, repo_id=f'YOUR_USER/recurrentlens-mamba130m-L{LAYER}-sae')
# print('uploaded to:', url)

## Open a PR

Once pushed, open a PR on `hinanohart/recurrentlens` adding the Hub ID to the README's "Pretrained SAEs" table. This is the community contribution path for v0.1.1.